# Bước 1: Khai báo thư viện và Đọc dữ liệu JSONL (Tập Train)

In [ ]:
import json
import os
import random
import math
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from PIL import Image
from io import BytesIO
import hashlib

# ============================================================
# Hàm valid_bbox - Sao chép từ scripts/prepare_swg.py (TV1)
# Dùng chung quy tắc kiểm tra Bounding Box với toàn nhóm
# ============================================================
def valid_bbox(ann, image):
    if not image:
        return None, 'missing_image'
    if ann.get('sequence_level_annotation'):
        return None, 'sequence_level_box'
    if 'bbox' in ann and 'bbox_relative' in ann:
        return None, 'ambiguous_coordinate_system'
    box = ann.get('bbox', ann.get('bbox_relative'))
    if not isinstance(box, list) or len(box) != 4:
        return None, 'no_bbox'
    if not all(isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v) for v in box):
        return None, 'nonfinite_bbox'
    w, h = image.get('width', 0), image.get('height', 0)
    if w <= 0 or h <= 0:
        return None, 'invalid_image_size'
    x, y, bw, bh = box
    if 'bbox_relative' in ann:
        x, y, bw, bh = x*w, y*h, bw*w, bh*h
    if min(x, y) < 0 or x >= w or y >= h or min(bw, bh) <= 0 or x+bw > w+1 or y+bh > h+1:
        return None, 'outside_or_nonpositive_bbox'
    return [x, y, min(bw, w-x), min(bh, h-y)], 'valid'

# ============================================================
# Tạo thư mục reports/ để lưu kết quả tự động
# ============================================================
REPORTS_DIR = "../reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

# ============================================================
# 1. Đọc dữ liệu từ tập TRAIN MANIFEST (Tránh Data Leakage)
# ============================================================
manifest_path = '../data/processed/v1/train.jsonl'
train_data = []

with open(manifest_path, 'r', encoding='utf-8') as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Tổng số ảnh trong tập Train: {len(train_data)}")

# Xây dựng từ điển tra cứu nhanh
images_dict = {img['image_id']: img for img in train_data}

categories_set = set(img['label'] for img in train_data)
print(f"Tổng số loài trong tập Train: {len(categories_set)}")


# Bước 2: Quét lỗi Bounding Box và Nhãn (Trên dữ liệu thô)

Thống kê quét toàn bộ metadata gốc để đồng bộ số liệu với quy tắc của hàm `valid_bbox` (TV1)

In [ ]:
# Bước 2 này chỉ chạy được khi bạn đã tải file ZIP metadata gốc về máy.
# Nếu chưa có file, cell sẽ tự bỏ qua và in thông báo.
# File zip: swg_camera_traps.bounding_boxes.with_species.zip
# Tải tại: https://lila.science/datasets/swg-camera-traps

import zipfile

RAW_ZIP = '../data/raw/swg_camera_traps.bounding_boxes.with_species.zip'

corrupted_images = []
missing_bbox_records = 0
bad_geometry_boxes = 0
valid_boxes_count = 0

try:
    import ijson  # Import bên trong try để không crash nếu chưa cài

    with zipfile.ZipFile(RAW_ZIP, 'r') as z:
        names = [n for n in z.namelist() if n.endswith('.json')]
        with z.open(names[0]) as f:
            # Lấy thông tin image gốc để test bbox
            print("Đang load thông tin ảnh gốc...")
            raw_images = {}
            for img in ijson.items(f, 'images.item'):
                raw_images[img['id']] = img
                if img.get('corrupt', False):
                    corrupted_images.append(img['id'])

            f.seek(0)
            print("Đang quét annotation...")
            for ann in ijson.items(f, 'annotations.item'):
                img_id = ann.get('image_id')
                img_info = raw_images.get(img_id)

                # Sử dụng hàm CHUẨN của nhóm (đã định nghĩa ở Bước 1)
                box, status = valid_bbox(ann, img_info)

                if status == 'valid':
                    valid_boxes_count += 1
                elif status == 'no_bbox':
                    missing_bbox_records += 1
                elif status in ['nonfinite_bbox', 'outside_or_nonpositive_bbox', 'invalid_image_size']:
                    bad_geometry_boxes += 1

    print("=== TỔNG KẾT QUÉT METADATA GỐC THEO QUY TẮC TV1 ===")
    print(f"1. Số ảnh đánh dấu hỏng (corrupt=True): {len(corrupted_images)}")
    print(f"2. Số bản ghi thiếu bbox (no_bbox): {missing_bbox_records}")
    print(f"3. Số Bounding Box lỗi hình học: {bad_geometry_boxes}")
    print(f"4. Bounding Box được chấp nhận: {valid_boxes_count}")

except ModuleNotFoundError:
    print("[BỎ QUA] Chưa cài thư viện 'ijson'. Chạy: pip install ijson để dùng bước này.")
except FileNotFoundError:
    print("[BỎ QUA] Không tìm thấy file ZIP metadata gốc. Tải về từ LILA BC rồi đặt vào data/raw/")
except Exception as e:
    print(f"[BỎ QUA] Lỗi không mong đợi: {e}")


# Bước 3: Hàm vẽ Bounding Box (Kiểm tra mẫu hợp lệ trong Train)

In [ ]:
def plot_image_with_boxes(image_id, save_path=None):
    img_info = images_dict[image_id]
    url = img_info['url']

    # Tải ảnh từ URL
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            print(f"Không thể tải ảnh: {url}")
            return
    except:
        print(f"Lỗi kết nối khi tải ảnh: {url}")
        return

    img = Image.open(BytesIO(response.content))
    fig, ax = plt.subplots(1, figsize=(10, 6))
    ax.imshow(img)

    boxes = img_info.get('boxes', [])
    for b in boxes:
        x, y, w, h = b['bbox_xywh'] # Đã chuẩn hóa pixel theo manifest
        cat_name = b['label']
        
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, max(0, y - 10), str(cat_name), color='red', fontsize=12, backgroundcolor='white')

    plt.title(f"Image ID: {image_id}\nLoc: {img_info.get('location')} | Split: {img_info.get('split')}")
    plt.axis('off')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"  >> Đã lưu: {save_path}")

    plt.show()

# Hiển thị thử 3 ảnh trong tập Train
bbox_check_dir = os.path.join(REPORTS_DIR, "bbox_checks")
os.makedirs(bbox_check_dir, exist_ok=True)

test_ids = [train_data[i]['image_id'] for i in [0, 3, 20]]
for idx, img_id in enumerate(test_ids):
    save_path = os.path.join(bbox_check_dir, f"check_bbox_train_{idx}.png")
    plot_image_with_boxes(img_id, save_path=save_path)


# Bước 4: Lấy mẫu 10 ảnh / loài cho 8 loài mục tiêu (Chỉ lấy từ TRAIN)

In [ ]:
TARGET_SPECIES = [
    'large_antlered_muntjac', 'annamite_striped_rabbit', 'sambar', 'chinese_serow',
    'common_palm_civet', 'masked_palm_civet', 'silver_pheasant', 'eurasian_wild_pig'
]

# Random cố định seed để tái lập kết quả
random.seed(42)

error_log = {
    "missing_or_http_errors": [],
    "duplicates_found": []
}
downloaded_hashes = {}
sample_audit_log = []

for species_name in TARGET_SPECIES:
    species_dir = os.path.join(REPORTS_DIR, species_name)
    os.makedirs(species_dir, exist_ok=True)

    # Gom nhóm theo location ĐỂ TĂNG TÍNH ĐA DẠNG (TRONG TẬP TRAIN)
    loc_to_imgs = defaultdict(list)
    for img in train_data:
        if img['label'] == species_name and len(img.get('boxes', [])) > 0:
            loc = img.get('location')
            loc_to_imgs[loc].append(img)
    
    sampled_imgs = []
    
    # Chiến lược lấy đa dạng địa điểm:
    locations = list(loc_to_imgs.keys())
    random.shuffle(locations) # Xáo trộn location
    
    idx_loc = 0
    while len(sampled_imgs) < 10 and len(locations) > 0:
        loc = locations[idx_loc % len(locations)]
        if len(loc_to_imgs[loc]) > 0:
            # Rút 1 ảnh ra khỏi location này
            img_choice = random.choice(loc_to_imgs[loc])
            loc_to_imgs[loc].remove(img_choice)
            sampled_imgs.append(img_choice)
        
        idx_loc += 1
        # Tránh vòng lặp vô tận nếu đã hết sạch ảnh
        if sum(len(lst) for lst in loc_to_imgs.values()) == 0:
            break

    print(f"\n[LOÀI: {species_name.upper()}] - Lấy {len(sampled_imgs)} mẫu từ tập Train.")

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    axes = axes.flatten()

    for idx, img_info in enumerate(sampled_imgs):
        ax = axes[idx]
        img_id = img_info['image_id']
        url = img_info['url']
        loc = img_info.get('location')
        seq = img_info.get('sequence_id')
        
        # Lưu audit
        sample_audit_log.append({
            "species": species_name,
            "image_id": img_id,
            "location": loc,
            "sequence_id": seq,
            "split": "train"
        })

        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                image_bytes = resp.content
                sha256_hash = hashlib.sha256(image_bytes).hexdigest()

                if sha256_hash in downloaded_hashes:
                    error_log["duplicates_found"].append({
                        "img_id_1": downloaded_hashes[sha256_hash],
                        "img_id_2": img_id,
                        "sha256": sha256_hash
                    })
                downloaded_hashes[sha256_hash] = img_id

                img = Image.open(BytesIO(image_bytes))
                single_img_path = os.path.join(species_dir, f"{idx+1:02d}_{img_id[-8:]}.jpg")
                img.save(single_img_path)

                ax.imshow(img)
                
                for b in img_info.get('boxes', []):
                    x, y, w, h = b['bbox_xywh']
                    rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)

                ax.set_title(f"Loc: {loc}\nSeq: ...{seq[-4:]}", fontsize=9)
            else:
                error_log["missing_or_http_errors"].append({"img_id": img_id, "error": resp.status_code})
                ax.text(0.5, 0.5, 'Lỗi HTTP', ha='center', va='center')
        except Exception as e:
            error_log["missing_or_http_errors"].append({"img_id": img_id, "error": str(e)})
            ax.text(0.5, 0.5, 'Lỗi kết nối', ha='center', va='center')

        ax.axis('off')

    # Ẩn các ô còn thừa nếu lấy không đủ 10 mẫu
    for i in range(len(sampled_imgs), 10):
        axes[i].axis('off')

    plt.suptitle(species_name.replace("_", " ").title(), fontsize=14, fontweight='bold')
    plt.tight_layout()

    grid_path = os.path.join(species_dir, f"grid_{species_name}.png")
    plt.savefig(grid_path, bbox_inches='tight', dpi=150)
    plt.close(fig) # Tắt vẽ để giải phóng RAM

# 4. LƯU BÁO CÁO
with open(os.path.join(REPORTS_DIR, "bien_ban_loi_w1.json"), "w", encoding="utf-8") as f:
    json.dump(error_log, f, indent=4, ensure_ascii=False)

with open(os.path.join(REPORTS_DIR, "sampled_audit.json"), "w", encoding="utf-8") as f:
    json.dump(sample_audit_log, f, indent=4, ensure_ascii=False)

print("=== HOÀN TẤT ===")
print("Đã lưu biên bản lỗi và file sample_audit.json vào thư mục reports/")
